In [3]:
import subprocess

result = subprocess.run(
    ["powershell.exe", "-Command", "nvidia-smi"],
    capture_output=True,
    text=True,
    shell=True,
)
print(result.stdout)

Wed Nov 11 18:16:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.36                 Driver Version: 566.36         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060      WDDM  |   00000000:09:00.0  On |                  N/A |
|  0%   50C    P0             43W /  180W |    1413MiB /  12288MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Нам интересна строка CUDA Version: 12.7 -> надо выполнить pip install cupy-cuda12x 

In [4]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount())      # должно быть >= 1
print(cp.cuda.runtime.getDeviceProperties(0)['name'])

1
b'NVIDIA GeForce RTX 3060'


In [2]:
print(cp.cuda.runtime.getDeviceProperties(0))

{'name': b'NVIDIA GeForce RTX 3060', 'totalGlobalMem': 12884246528, 'sharedMemPerBlock': 49152, 'regsPerBlock': 65536, 'warpSize': 32, 'maxThreadsPerBlock': 1024, 'maxThreadsDim': (1024, 1024, 64), 'maxGridSize': (2147483647, 65535, 65535), 'clockRate': 1837000, 'totalConstMem': 65536, 'major': 8, 'minor': 6, 'textureAlignment': 512, 'texturePitchAlignment': 32, 'multiProcessorCount': 28, 'kernelExecTimeoutEnabled': 1, 'integrated': 0, 'canMapHostMemory': 1, 'computeMode': 0, 'maxTexture1D': 131072, 'maxTexture2D': (131072, 65536), 'maxTexture3D': (16384, 16384, 16384), 'concurrentKernels': 1, 'ECCEnabled': 0, 'pciBusID': 9, 'pciDeviceID': 0, 'pciDomainID': 0, 'tccDriver': 0, 'memoryClockRate': 7501000, 'memoryBusWidth': 192, 'l2CacheSize': 2359296, 'maxThreadsPerMultiProcessor': 1536, 'isMultiGpuBoard': 0, 'cooperativeLaunch': 1, 'cooperativeMultiDeviceLaunch': 0, 'maxTexture1DMipmap': 32768, 'maxTexture1DLayered': (32768, 2048), 'maxTexture2DMipmap': (32768, 32768), 'maxTexture2DLine

вывод первой стороки 1 или более - это количество видеоадаптеров. если 0 - все печально.
вывод второй строки говорит о том, что код понимает, какая карточка стоит

In [7]:
# -*- coding: utf-8 -*-
"""
mnist_cupy_batch.py
===================
Та же сеть, что и в mnist_numpy_batch.py, но все вычисления на GPU NVIDIA
через CuPy. Логика и API идентичны, отличаются только:
    * numpy -> cupy, scipy.special.expit -> cupyx.scipy.special.expit
    * dtype по умолчанию float32 (на GeForce float64 в 32-64 раза медленнее)
    * данные один раз переносятся в память GPU
    * joblib сохраняет/загружает через cp.asnumpy / cp.asarray
    * замеры времени — только с deviceSynchronize()

Установка (смотри версию CUDA в nvidia-smi):
    pip install cupy-cuda12x     # CUDA 12.x
    pip install cupy-cuda13x     # CUDA 13.x
НЕ ставь "pip install cupy" — это сборка из исходников.

Формат весов {'ih', 'ho'} совместим с NumPy-версией.
"""

import os
import time

import joblib
import numpy as np                      # нужен только для чтения CSV с диска

try:
    import cupy as cp
    from cupyx.scipy.special import expit
except ImportError as e:
    raise ImportError(
        "Не найден CuPy. Поставь wheel под свою CUDA, например:\n"
        "    pip install cupy-cuda12x\n"
        "Версию CUDA смотри в выводе nvidia-smi."
    ) from e


def _sync():
    """Дождаться завершения всех GPU-операций (без этого замеры времени врут)."""
    cp.cuda.runtime.deviceSynchronize()


class MNIST_Neural_Network:
    """Трёхслойняя сеть с одним скрытым слоем, батчевое обучение на GPU."""

    def __init__(self,
                 input_nodes=784,
                 hidden_nodes=200,
                 output_nodes=10,
                 learning_rate=0.5,
                 epochs=10,
                 batch_size=64,
                 grad_reduction="mean",   # "mean" | "sum"
                 shuffle=True,
                 seed=None,
                 dtype=cp.float32,
                 device_id=0):
        if grad_reduction not in ("mean", "sum"):
            raise ValueError("grad_reduction должен быть 'mean' или 'sum'")
        if batch_size < 1:
            raise ValueError("batch_size должен быть >= 1")

        self.device = cp.cuda.Device(device_id)
        self.device.use()

        self.input_nodes = int(input_nodes)
        self.hidden_nodes = int(hidden_nodes)
        self.output_nodes = int(output_nodes)
        self.learning_rate = float(learning_rate)
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.grad_reduction = grad_reduction
        self.shuffle = bool(shuffle)
        self.dtype = cp.dtype(dtype)

        # индексы перемешивания генерим на CPU, веса — на GPU
        self._rng = np.random.default_rng(seed)
        self._cp_rng = cp.random.RandomState(seed)

        self.weights_input_hidden = self._cp_rng.normal(
            0.0, pow(self.input_nodes, -0.5),
            (self.hidden_nodes, self.input_nodes)
        ).astype(self.dtype)

        self.weights_hidden_output = self._cp_rng.normal(
            0.0, pow(self.hidden_nodes, -0.5),
            (self.output_nodes, self.hidden_nodes)
        ).astype(self.dtype)

        # ссылка на функцию, а не lambda
        self.activation_function = expit

    # ------------------------------------------------------------------ #
    #  Работа с данными
    # ------------------------------------------------------------------ #
    def _asarray(self, x):
        """numpy/list -> GPU; если уже GPU-массив нужного типа — без копии."""
        return cp.asarray(x, dtype=self.dtype)

    def _prepare_arrays(self, data):
        """data (N, 1+input_nodes) на CPU -> X, T на GPU (столбцы = примеры)."""
        data = np.asarray(data, dtype=np.float64)
        if data.ndim == 1:
            data = data.reshape(1, -1)

        expected = self.input_nodes + 1
        if data.shape[1] != expected:
            raise ValueError(
                f"Ожидалось {expected} столбцов (1 метка + {self.input_nodes} пикселей), "
                f"получено {data.shape[1]}"
            )

        labels = data[:, 0].astype(np.int64)
        if labels.min() < 0 or labels.max() >= self.output_nodes:
            raise ValueError("Метка выходит за диапазон output_nodes")

        X = (data[:, 1:] / 255.0 * 0.99) + 0.01
        T = np.full((labels.size, self.output_nodes), 0.01, dtype=np.float64)
        T[np.arange(labels.size), labels] = 0.99

        # ОДИН перенос всего датасета в память видеокарты
        X = cp.ascontiguousarray(cp.asarray(X.T, dtype=self.dtype))
        T = cp.ascontiguousarray(cp.asarray(T.T, dtype=self.dtype))
        return X, T

    def _load_dataset(self, path):
        data = np.loadtxt(path, delimiter=",", dtype=np.float64)
        return self._prepare_arrays(data)

    # ------------------------------------------------------------------ #
    #  Обучение
    # ------------------------------------------------------------------ #
    def train_batch(self, X, T):
        """
        Один шаг обучения на батче (всё на GPU, без синхронизации).
        X: (input_nodes, batch), T: (output_nodes, batch).
        Возвращает MSE как 0-мерный GPU-массив (не скаляр! — чтобы не тормозить).
        """
        X = self._asarray(X)
        T = self._asarray(T)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        if T.ndim == 1:
            T = T.reshape(-1, 1)

        if X.shape[0] != self.input_nodes or T.shape[0] != self.output_nodes:
            raise ValueError("Неверная форма X или T")
        if X.shape[1] != T.shape[1]:
            raise ValueError("В X и T разное количество примеров")

        n = X.shape[1]

        H = self.activation_function(self.weights_input_hidden @ X)   # (hidden, n)
        O = self.activation_function(self.weights_hidden_output @ H)  # (output, n)

        E_out = T - O
        E_hid = self.weights_hidden_output.T @ E_out   # по СТАРЫМ весам

        scale = self.learning_rate / n if self.grad_reduction == "mean" else self.learning_rate

        self.weights_hidden_output += scale * ((E_out * O * (1.0 - O)) @ H.T)
        self.weights_input_hidden  += scale * ((E_hid * H * (1.0 - H)) @ X.T)

        return cp.mean(E_out * E_out)   # остаётся на GPU

    def train(self, training_list_path="mnist_train_100.csv", verbose=True):
        X, T = self._load_dataset(training_list_path)
        n = X.shape[1]
        n_batches = (n + self.batch_size - 1) // self.batch_size

        if verbose:
            name = cp.cuda.runtime.getDeviceProperties(self.device.id)["name"]
            print(f"[GPU] {name} | dtype={self.dtype.name} | примеров: {n} | "
                  f"батч: {self.batch_size} | батчей/эпоху: {n_batches}")

        _sync()
        t0 = time.perf_counter()

        for epoch in range(1, self.epochs + 1):
            order_np = self._rng.permutation(n) if self.shuffle else np.arange(n)
            order = cp.asarray(order_np)          # маленький трансфер индексов

            epoch_loss = cp.zeros((), dtype=self.dtype)   # копим на GPU
            for start in range(0, n, self.batch_size):
                idx = order[start:start + self.batch_size]
                loss = self.train_batch(X[:, idx], T[:, idx])
                epoch_loss += loss * idx.size

            if verbose:
                # float() -> единственная синхронизация за эпоху
                print(f"  epoch {epoch}/{self.epochs}  "
                      f"MSE = {float(epoch_loss) / n:.6f}  "
                      f"[{time.perf_counter() - t0:.2f} s]")

        _sync()
        if verbose:
            print(f"[GPU] обучение завершено за {time.perf_counter() - t0:.3f} c")

        return self.weights_input_hidden, self.weights_hidden_output

    # ------------------------------------------------------------------ #
    #  Предсказания
    # ------------------------------------------------------------------ #
    def query(self, inputs_list):
        """Один пример: (784,) -> (10, 1) на GPU."""
        X = self._asarray(inputs_list)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        H = self.activation_function(self.weights_input_hidden @ X)
        return self.activation_function(self.weights_hidden_output @ H)

    def query_batch(self, X):
        """Батч: X (input_nodes, batch) -> (output_nodes, batch) на GPU."""
        X = self._asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        H = self.activation_function(self.weights_input_hidden @ X)
        return self.activation_function(self.weights_hidden_output @ H)

    def predict(self, samples):
        """samples (N, 784) строками -> метки (N,) numpy на CPU."""
        S = self._asarray(samples)
        if S.ndim == 1:
            S = S.reshape(1, -1)
        return cp.asnumpy(cp.argmax(self.query_batch(S.T), axis=0))

    def return_answers(self, input_data):
        out = cp.asnumpy(self.query(input_data)).ravel()   # один трансфер 10 чисел
        idx = int(np.argmax(out))
        return idx, float(out[idx]) * 100.0

    # ------------------------------------------------------------------ #
    #  Оценка качества
    # ------------------------------------------------------------------ #
    def score_info(self, test_data="mnist_test_10.csv", verbose=True):
        if isinstance(test_data, (str, os.PathLike)):
            X, T = self._load_dataset(test_data)
        else:
            rows = [[float(v) for v in rec.split(",")] for rec in test_data]
            X, T = self._prepare_arrays(rows)

        labels = cp.argmax(T, axis=0)
        n = X.shape[1]
        correct = cp.zeros((), dtype=cp.int64)    # копим на GPU, без синхронизаций

        for start in range(0, n, self.batch_size):
            sl = slice(start, start + self.batch_size)
            pred = cp.argmax(self.query_batch(X[:, sl]), axis=0)
            correct += cp.count_nonzero(pred == labels[sl])

        correct = int(correct)                    # одна синхронизация в конце
        acc = correct / n
        if verbose:
            print(f"performance = {acc}  ({correct}/{n})")
        return acc

    # ------------------------------------------------------------------ #
    #  Сохранение / загрузка весов (GPU <-> CPU <-> файл)
    # ------------------------------------------------------------------ #
    def save_weights(self, file_name: str):
        joblib.dump({'ih': cp.asnumpy(self.weights_input_hidden),
                     'ho': cp.asnumpy(self.weights_hidden_output)}, file_name)

    def load_weights(self, file_name: str):
        w = joblib.load(file_name)
        ih = cp.asarray(w['ih'], dtype=self.dtype)
        ho = cp.asarray(w['ho'], dtype=self.dtype)

        if ih.shape != (self.hidden_nodes, self.input_nodes):
            raise ValueError(f"W_ih: ожидалось {(self.hidden_nodes, self.input_nodes)}, получено {ih.shape}")
        if ho.shape != (self.output_nodes, self.hidden_nodes):
            raise ValueError(f"W_ho: ожидалось {(self.output_nodes, self.hidden_nodes)}, получено {ho.shape}")

        self.weights_input_hidden = ih
        self.weights_hidden_output = ho

    # ------------------------------------------------------------------ #
    #  Картинка -> ответ
    # ------------------------------------------------------------------ #
    def input_from_picture(self, picture_path, invert=True):
        from PIL import Image, ImageOps  # pip install pillow

        side = int(round(self.input_nodes ** 0.5))
        img = Image.open(picture_path).convert("L")
        if img.size != (side, side):
            img = img.resize((side, side), Image.LANCZOS)
        if invert:
            img = ImageOps.invert(img)

        pixels = np.asarray(img.getdata(), dtype=np.float64)
        pixels = (pixels / 255.0 * 0.99) + 0.01
        return self.return_answers(pixels)


# ====================================================================== #
#  Демонстрация
# ====================================================================== #
if __name__ == "__main__":
    TRAIN_FILE = "mnist_train.csv"
    TEST_FILE = "mnist_test.csv"
    WEIGHTS_FILE = "weights_cupy_batch.joblib"

    net = MNIST_Neural_Network(
        input_nodes=784, hidden_nodes=200, output_nodes=10,
        learning_rate=0.5,
        epochs=300,
        batch_size=256,
        grad_reduction="mean",
        seed=42,
        dtype=cp.float32,   # на GeForce обязательно float32
        device_id=0,
    )

    if os.path.exists(TRAIN_FILE):
        net.train(TRAIN_FILE)
        net.save_weights(WEIGHTS_FILE)
        print(f"веса сохранены -> {WEIGHTS_FILE}")
    else:
        print(f"нет файла {TRAIN_FILE} — обучение пропущено")
        if os.path.exists(WEIGHTS_FILE):
            net.load_weights(WEIGHTS_FILE)
            print(f"веса загружены <- {WEIGHTS_FILE}")

    if os.path.exists(TEST_FILE):
        net.score_info(TEST_FILE)

[GPU] b'NVIDIA GeForce RTX 3060' | dtype=float32 | примеров: 60000 | батч: 256 | батчей/эпоху: 235
  epoch 1/300  MSE = 0.046731  [0.15 s]
  epoch 2/300  MSE = 0.025114  [0.28 s]
  epoch 3/300  MSE = 0.020338  [0.42 s]
  epoch 4/300  MSE = 0.018046  [0.56 s]
  epoch 5/300  MSE = 0.016624  [0.70 s]
  epoch 6/300  MSE = 0.015570  [0.87 s]
  epoch 7/300  MSE = 0.014684  [1.02 s]
  epoch 8/300  MSE = 0.013938  [1.16 s]
  epoch 9/300  MSE = 0.013282  [1.29 s]
  epoch 10/300  MSE = 0.012658  [1.44 s]
  epoch 11/300  MSE = 0.012173  [1.58 s]
  epoch 12/300  MSE = 0.011664  [1.71 s]
  epoch 13/300  MSE = 0.011200  [1.85 s]
  epoch 14/300  MSE = 0.010802  [1.98 s]
  epoch 15/300  MSE = 0.010470  [2.12 s]
  epoch 16/300  MSE = 0.010152  [2.25 s]
  epoch 17/300  MSE = 0.009863  [2.39 s]
  epoch 18/300  MSE = 0.009582  [2.54 s]
  epoch 19/300  MSE = 0.009300  [2.68 s]
  epoch 20/300  MSE = 0.009040  [2.83 s]
  epoch 21/300  MSE = 0.008803  [2.99 s]
  epoch 22/300  MSE = 0.008579  [3.14 s]
  epoch 